In [12]:
import pandas as pd

partners = ["826", "MB", "EF", "GNL"]


class Profile:
    def __init__(self, profile: dict | None = None):
        if profile is not None:
            self.profile = profile
        else:
            profile = {}

    def __getitem__(self, key):
        return self.profile.get(key, False)

    def __setitem__(self, key, value):
        self.profile[key] = value


def exclusions(profile):
    excluded = {}

    # Mission Bit requires HS interest.
    if not profile["grade_high"]:
        excluded.setdefault("MB", []).append(
            "Mission Bit requires interest in high-school students"
        )

    # Education Fund requires weekday availability.
    weekday = any([
        profile["avail_weekday_day_inperson"],
        profile["avail_weekday_after_school_inperson"],
        profile["avail_weekday_day_remote"],
        profile["avail_weekday_after_school_remote"],
        profile["avail_weekday_evening"],
        profile["avail_flexible"],
    ])

    if not weekday:
        excluded.setdefault("EF", []).append(
            "Education Fund requires weekday availability"
        )

    remote_choices = [
        profile["avail_weekday_day_remote"],
        profile["avail_weekday_after_school_remote"],
    ]

    inperson_choices = [
        profile["avail_weekday_day_inperson"],
        profile["avail_weekday_after_school_inperson"],
        profile["avail_weekend_inperson"],
    ]

    remote_only = any(remote_choices) and not any(inperson_choices)

    if remote_only:
        excluded.setdefault("826", []).append(
            "826 does not support remote-only availability"
        )
        excluded.setdefault("EF", []).append(
            "Education Fund does not support remote-only availability"
        )

    direct = (
        profile["work_direct"]
        or profile["work_classroom"]
    )

    if not direct:
        for partner in ["MB", "GNL", "826"]:
            excluded.setdefault(partner, []).append(
                "Requires willingness to work directly with students/in classroom"
            )

    return excluded


weights = pd.DataFrame.from_dict({
    # Interests
    "interest_stem":             [0, 5, 1, 0],
    "interest_career":           [1, 3, 1, 0],
    "interest_mentoring":        [1, 1, 5, 0],
    "interest_events":           [1, 3, 1, 1],
    "interest_gardening":        [0, 0, 0, 0],
    "interest_writing":          [5, 1, 3, 0],
    "interest_admin":            [0, 1, 1, 5],
    "interest_flexible":         [1, 1, 2, 5],
    "interest_maintenance":      [0, 0, 0, 0],
    "interest_sports":           [0, 0, 0, 0],
    "interest_other":            [0, 0, 0, 0],

    # Grade
    "grade_elementary":          [5, 0, 3, 1],
    "grade_middle":              [5, 1, 3, 1],
    "grade_high":                [4, 5, 3, 1],

    # Student interaction
    "work_direct":          [5, 2, 5, 0],
    "work_classroom":            [4, 5, 5, 0],
    "work_behind_scenes":        [0, 3, 1, 5],

    # Availability
    "avail_weekday_day_inperson": [4, 1, 5, 2],
    "avail_weekday_3_6_inperson": [4, 5, 0, 2],
    "avail_weekday_day_remote":  [0, 1, 0, 4],
    "avail_weekday_3_6_remote":  [0, 5, 0, 4],
    "avail_weekend_inperson":    [4, 1, 0, 2],
    "avail_weekday_after_6":     [0, 0, 0, 3],
    "avail_flexible":            [2, 2, 3, 5],

    # Frequency
    "frequency_weekly":          [3, 3, 5, 2],
    "frequency_monthly":         [2, 3, 0, 3],
    "frequency_adhoc":           [2, 5, 0, 3],
    "frequency_unsure":          [1, 1, 1, 2],
}, orient="index", columns=partners)

groups = {
    "interests": [
        c for c in weights.index if c.startswith("interest_")
    ],
    "grades": [
        c for c in weights.index if c.startswith("grade_")
    ],
    "work_style": [
        c for c in weights.index if c.startswith("work_")
    ],
    "availability": [
        c for c in weights.index if c.startswith("avail_")
    ],
    "frequency": [
        c for c in weights.index if c.startswith("frequency_")
    ],
}

group_importance = {
    "interests": 0.45,
    "grades": 0.15,
    "work_style": 0.15,
    "availability": 0.15,
    "frequency": 0.10,
}


def group_score(profile, partner, features):
    selected = [f for f in features if profile.get(f, False)]

    if not selected:
        return 0

    values = weights.loc[selected, partner]

    # 5 represents a "strong match", so normalize to 0..1.
    return values.mean() / 5


def score_partner(profile, partner):
    components = {}

    for group, features in groups.items():
        components[group] = group_score(
            profile,
            partner,
            features
        )

    total = sum(
        components[group] * group_importance[group]
        for group in components
    )

    return total, components


def evaluate(profile):
    blocked = exclusions(profile)

    results = []

    for partner in partners:
        if partner in blocked:
            results.append({
                "partner": partner,
                "eligible": False,
                "score": None,
                "excluded_by": blocked[partner],
            })
            continue

        score, components = score_partner(profile, partner)

        results.append({
            "partner": partner,
            "eligible": True,
            "score": score,
            "components": components,
        })

    return pd.DataFrame(results).sort_values(
        "score",
        ascending=False,
        na_position="last",
    )

In [11]:
# fake person
alex = {
    "interest_stem": True,
    "interest_career": True,
    "grade_high": True,
    "work_classroom": True,
    "avail_weekday_3_6_inperson": True,
    "frequency_weekly": True,
}

evaluate(alex)

KeyError: 'avail_weekday_day_inperson'